![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)


# M3L2 E17 - Conectar LangChain a un servidor local: arquitectura desacoplada (Resolution)

## Qué es este notebook

En E16 vimos las piezas conceptuales (modelo, motor, servidor, wrapper). Acá las conectamos: apuntamos `ChatOpenAI` a un servidor **local** (LM Studio, puerto `1234`) en vez de a la API de OpenAI, y armamos un **model factory** que permite cambiar de proveedor (OpenAI, LM Studio, Ollama, vLLM) cambiando una sola variable de entorno — sin tocar el resto del pipeline.

Las celdas que requieren un servidor local corriendo están marcadas como **opcionales**: se saltean solas con un mensaje si no hay nada escuchando en ese puerto. El resto del notebook funciona con tu `OPENAI_API_KEY` de siempre, para que puedas ejecutarlo sin instalar nada extra.


In [ ]:
# %pip (no !pip): instala en el Python del kernel activo, no depende del PATH de tu shell.
%pip install langchain-openai
# Opcional (BLOQUE 2, provider="ollama"): %pip install langchain-ollama


In [ ]:
import os, getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")


## BLOQUE 1 — Un servidor local también habla "OpenAI"

LM Studio expone un servidor compatible con la API de OpenAI en `http://localhost:1234/v1`. Eso significa que podemos seguir usando `ChatOpenAI` — solo cambian tres parámetros:

```text
model    -> el identificador que LM Studio le puso a tu modelo cargado
base_url -> http://localhost:1234/v1  (en vez de la API de OpenAI)
api_key  -> cualquier string no vacio (LM Studio no valida la key)
```

```text
LangChain
   ↓
ChatOpenAI
   ↓
http://localhost:1234/v1
   ↓
LM Studio
   ↓
Modelo local (GGUF)
```


In [ ]:
# Conexion a LM Studio (opcional). Se saltea sola si no tenes LM Studio corriendo.
# 1) Abrir LM Studio -> Developer -> Start Server
# 2) Cargar un modelo instruct

from langchain_openai import ChatOpenAI

try:
    llm_lmstudio = ChatOpenAI(
        model="local-model",
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
        temperature=0.2,
        timeout=10,
        max_retries=1,
    )
    respuesta = llm_lmstudio.invoke("Di solo la palabra: hola")
    print(f"LM Studio responde: {respuesta.content}")
except Exception as e:
    print("Salteado: no se pudo conectar a LM Studio en localhost:1234.")
    print(f"Detalle: {e}")


> Si `curl http://localhost:1234/v1/models` falla, el problema está en LM Studio (no está corriendo, no cargó modelo, puerto distinto) — no en LangChain ni en Python. Separar el problema así ahorra mucho tiempo de debugging.


## BLOQUE 2 — Model factory: desacoplar la elección de proveedor

No conviene instanciar el modelo (con su `base_url`, `api_key`, `model`) en cada archivo. Centralizamos esa decisión en una única función, controlada por una variable de entorno.

```text
MODEL_PROVIDER=openai     -> ChatOpenAI apuntando a la API real
MODEL_PROVIDER=lmstudio   -> ChatOpenAI apuntando a localhost:1234
MODEL_PROVIDER=ollama     -> ChatOllama
MODEL_PROVIDER=vllm       -> ChatOpenAI apuntando a un servidor vLLM propio
```


In [ ]:
def create_chat_model(provider: str = "openai"):
    """Devuelve un BaseChatModel segun el proveedor pedido. Un solo lugar para cambiar de proveedor."""
    from langchain_openai import ChatOpenAI

    provider = provider.lower()

    if provider == "openai":
        return ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

    if provider == "lmstudio":
        return ChatOpenAI(
            model="local-model",
            base_url="http://localhost:1234/v1",
            api_key="lm-studio",
            temperature=0.1,
            timeout=10,
            max_retries=1,
        )

    if provider == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model="llama3.2", temperature=0.1)

    if provider == "vllm":
        return ChatOpenAI(
            model="modelo-servido",
            base_url=os.environ["VLLM_BASE_URL"],
            api_key=os.environ["VLLM_API_KEY"],
            temperature=0.1,
        )

    raise ValueError(f"Proveedor no soportado: {provider}")


# El "path feliz" de este notebook: siempre funciona con tu OPENAI_API_KEY
model = create_chat_model(os.getenv("MODEL_PROVIDER", "openai"))
print(f"Proveedor activo: {os.getenv('MODEL_PROVIDER', 'openai')}")
print(f"Tipo de objeto: {type(model).__name__}")
print(model.invoke("Di solo la palabra: hola").content)


### El beneficio del desacoplamiento

La lógica de negocio no cambia nunca:

```python
chain = prompt | model | parser
```

Lo único que cambia es **una variable de entorno** (`MODEL_PROVIDER`), no una línea de código dentro de la lógica de la aplicación.

| Situación | Qué cambia |
|---|---|
| Pasar de OpenAI a LM Studio para desarrollo local | `MODEL_PROVIDER=lmstudio` |
| Pasar de LM Studio a un servidor vLLM en producción | `MODEL_PROVIDER=vllm` + `VLLM_BASE_URL`/`VLLM_API_KEY` |
| El resto del pipeline (`prompt`, `parser`, `chain`) | No cambia nada |

**Relacionado con**: E00 - LLM Wrapper (mismo patrón, ahora parametrizado).


## BLOQUE 3 — Qué puede necesitar ajustes al cambiar de modelo

La interfaz común (`.invoke()`) reduce el acoplamiento, pero **no hace idénticos a todos los modelos**. Al cambiar de proveedor puede hacer falta ajustar:

- System prompt (algunos modelos locales siguen instrucciones peor que GPT-4o-mini).
- Temperatura y `max_tokens`.
- Formato de herramientas / soporte de tool calling.
- Tamaño del contexto disponible.

```text
Misma interfaz != mismo comportamiento.
```


## BLOQUE 4 — Streaming

| Método | Comportamiento |
|---|---|
| `invoke()` | Espera la respuesta completa |
| `stream()` | Entrega fragmentos progresivamente |
| `batch()` | Ejecuta varias entradas en paralelo |
| `ainvoke()` | Versión asíncrona de `invoke()` |
| `astream()` | Streaming asíncrono |

El streaming funciona igual sin importar qué wrapper esté detrás — es parte de la interfaz común de `BaseChatModel`.


In [ ]:
for chunk in model.stream("Explica en 3 pasos como funciona un RAG."):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()


## BLOQUE 5 — Diseñar contra la interfaz, no contra la implementación

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def build_chain(chat_model: BaseChatModel):
    """El codigo de negocio depende de BaseChatModel, no de ChatOpenAI ni de ningun proveedor concreto."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Sos un profesor de ingenieria de IA. Respondes en una oracion."),
        ("human", "{pregunta}"),
    ])
    return prompt | chat_model | StrOutputParser()


chain = build_chain(model)
print(chain.invoke({"pregunta": "Que es un servidor de inferencia?"}))


## Resumen — Lo que demuestra E17

| Sin desacoplar | Con model factory |
|---|---|
| `ChatOpenAI(base_url=..., api_key=..., model=...)` repetido en cada archivo | `create_chat_model(provider)` en un solo lugar |
| Cambiar de proveedor = buscar y reemplazar en todo el proyecto | Cambiar de proveedor = una variable de entorno |
| El código de negocio conoce el proveedor concreto | El código de negocio solo conoce `BaseChatModel` |

**Relacionado con**: E00 - LLM Wrapper, E16 - Arquitectura de modelos open source.


## Checks automáticos

In [ ]:
def run_checks():
    from langchain_core.messages import AIMessage

    m = create_chat_model("openai")
    r = m.invoke("Di solo: test")
    assert isinstance(r, AIMessage)
    assert len(r.content) > 0

    resultado_chain = chain.invoke({"pregunta": "Que es LangChain?"})
    assert isinstance(resultado_chain, str)
    assert len(resultado_chain) > 0

    try:
        create_chat_model("proveedor-inexistente")
        assert False, "deberia haber lanzado ValueError"
    except ValueError:
        pass

    print("M3L2 E17 Resolution checks passed")


run_checks()
